# Lab: RAG Basics - Building Retrieval Pipeline

## Learning Objectives

By the end of this lab, you will be able to:
- Understand the core components of a retrieval pipeline
- Implement document chunking strategies
- Create and work with text embeddings
- Build a vector-based retrieval system
- Evaluate retrieval quality

## Introduction

Retrieval-Augmented Generation (RAG) is a technique that enhances language models by retrieving relevant information from a knowledge base before generating responses. The retrieval pipeline is the foundation of any RAG system, responsible for finding the most relevant information given a query.

In this lab, you will build a complete retrieval pipeline from scratch, learning each component step by step.


In [1]:
## Setup

### Required Libraries

# Install the following libraries:

!pip install sentence-transformers numpy scikit-learn

## Part 1: Document Preprocessing and Chunking

### Understanding the Importance

Before we can retrieve documents, we need to prepare them properly. In real-world scenarios, documents are often too large to process as single units. Chunking breaks documents into smaller, manageable pieces while preserving context.

**Why this matters:** Poor chunking can lead to lost context or irrelevant retrievals, directly impacting the quality of your RAG system.

In [115]:
# ----------------------------------------
## Task 1.1: Implement Basic Chunking
# ----------------------------------------

###Create a function that splits documents into chunks based on a maximum token/word count.
def chunk_by_words(text, max_words=50, overlap=10):
    """
    Split text into chunks based on word count.
    
    Args:
        text: Input text to chunk
        max_words: Maximum words per chunk
        overlap: Number of overlapping words between chunks
    
    Returns:
        List of text chunks
    """
    # Your code here
    # convert full text into list of words
    words = text.split()
    # empty list to store the final chunks
    chunks = []  
    # defining the step size
    step = max_words - overlap
    
    for i in range(0, len(words), step):
        # take max words from curent position
        chunk = words[i:i + max_words]

        # join words back into sentence
        chunk_text = " ".join(chunk)

        # add into chunk list
        chunks.append(chunk_text)
        # stop if last chunk is smaller than max_words
        if i +max_words >= len(words):
            break
        return chunks  

# -------------------------------
# **Testing the implementation:**
# --------------------------------

sample_text = "Machine learning is a subset of artificial intelligence. It enables systems to learn from experience. Deep learning uses neural networks with multiple layers."

chunks = chunk_by_words(sample_text, max_words=10, overlap=3)
print(f"Number of chunks: {len(chunks)}")
for i, chunk in enumerate(chunks):
    print(f"Chunk {i+1}: {chunk}")

Number of chunks: 1
Chunk 1: Machine learning is a subset of artificial intelligence. It enables


In [116]:
# ----------------------------------------
### Task 1.2: Sentence-Based Chunking
# ----------------------------------------

# Implement a more sophisticated chunking strategy that respects sentence boundaries.


def chunk_by_sentences(text, max_sentences=2):
    """
    Split text into chunks based on complete sentences.
    """
    # Your code here
    # split text into sentence, whenever there is (.)
    sentences= text.split(". ")

    chunks = []
    # loop through the sentence
    for i in range(0, len(sentences), max_sentences):

        # taking max sentence at time
        chunk = sentences[i:i + max_sentences]
        # join them back
        chunk_text= ". ".join(chunk)
        # Adding period if missing
        if not chunk_text.endswith("."):
            chunk_text +="."
        # store chunks in list
        chunks.append(chunk_text)
    return chunk

# --------------------------------
 # **Testing the implementation:**
# --------------------------------

sample_text = "Machine learning is a subset of artificial intelligence. It enables systems to learn from experience. Deep learning uses neural networks with multiple layers."

chunks = chunk_by_sentences(sample_text, max_sentences=2)
print(f"Number of chunks: {len(chunks)}")
for i, chunk in enumerate(chunks):
    print(f"Chunk {i+1}: {chunk}")   
    
    # Hint: Use simple sentence splitting on periods followed by spaces

# --------------------------------
# **Questions to consider:**
# --------------------------------

 # What are the advantages of sentence-based chunking over word-based chunking?
  # no broken sentences and keep the complete meaning, chunks are more meaningful

#  When might word-based chunking be more appropriate?
   # when we work with very long documents and sentence are extremly long and we want to control the exact token size
    

Number of chunks: 1
Chunk 1: Deep learning uses neural networks with multiple layers.



## Part 2: Creating Embeddings

### Understanding the Importance

Embeddings convert text into numerical vectors that capture semantic meaning. This transformation is crucial because computers can't directly compare text semantically, but they can measure distances between vectors.

**Why this matters:** The quality of embeddings directly determines how well your retrieval system understands semantic similarity. Poor embeddings will retrieve irrelevant documents even if your retrieval logic is perfect.


In [117]:
# ----------------------------------------
### Task 2.1: Initialize Embedding Model
# ----------------------------------------

# Load a pre-trained sentence transformer model.
# import the class neede to load embedding models
from sentence_transformers import SentenceTransformer

# all-MiniLM-L6-v2 -> small and fast for semantic search 
def initialize_embedding_model(model_name='all-MiniLM-L6-v2'):
    """
    Initialize a sentence transformer model for creating embeddings.
    
    Args:
        model_name: Name of the pre-trained model
    
    Returns:
        Loaded model
    """
    # Your code here
    
    # load pretrained model
    model = SentenceTransformer(model_name)

    # Return the loaded model
    return model
    

In [118]:
from documents import DOCUMENTS

In [119]:
# ----------------------------------------
### Task 2.2: Generate Document Embeddings
# ----------------------------------------

# Create embeddings for all documents in your knowledge base.

import numpy as np
def create_embeddings(documents, model):
    """
    Generate embeddings for a list of documents.
    
    Args:
        documents: List of text documents
        model: Embedding model
    
    Returns:
        numpy array of embeddings
    """
    # Your code here
    # converts documents into embeddings (vectors)
    embeddings= model.encode(documents)
    # convert embeddings into numpy array
    embeddings = np.array(embeddings)
    # return all embeddings
    return embeddings 

# ------------------------------
# **Testing the implementation:**
# ------------------------------

from documents import DOCUMENTS

model = initialize_embedding_model()
embeddings = create_embeddings(DOCUMENTS, model)

print(f"Number of documents: {len(DOCUMENTS)}")
print(f"Embedding shape: {embeddings.shape}")
print(f"First embedding (first 10 dimensions): {embeddings[0][:10]}")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Number of documents: 10
Embedding shape: (10, 384)
First embedding (first 10 dimensions): [ 0.00840924 -0.00360583  0.05522151  0.06061023  0.03103588 -0.02438612
 -0.02974017 -0.03058421 -0.07072206 -0.00281699]


In [120]:
# -----------------------------------------
### Task 2.3: Understanding Embedding Space
# -----------------------------------------

# Calculate and visualize the similarity between different documents.

import numpy as np

def cosine_similarity(vec1, vec2):
    """
    Calculate cosine similarity between two vectors.
    
    Args:
        vec1: First vector
        vec2: Second vector
    
    Returns:
        Similarity score between -1 and 1
    """
    # Your code here
    # Calculate dot product
    dot_product = np.dot(vec1, vec2)

    # Calculate magnitude (length) of each vector
    norm_vec1 = np.linalg.norm(vec1)
    norm_vec2 = np.linalg.norm(vec2)
    # Compute cosine similarity
    similarity = dot_product / (norm_vec1 * norm_vec2)

    # return similarity score
    return similarity
    


# -----------------
# **Experiment:**
# -----------------

# Compare similarities between different document pairs
# Document 0: Machine learning definition
# Document 1: Deep learning definition
# Document 4: Reinforcement learning definition

# comapres machine learning and deep earning
similarity_ml_dl = cosine_similarity(embeddings[0], embeddings[1])
# comapare machine learning and reinforcement learning
similarity_ml_rl = cosine_similarity(embeddings[0], embeddings[4])

print(f"Similarity (ML vs DL): {similarity_ml_dl:.4f}")
print(f"Similarity (ML vs RL): {similarity_ml_rl:.4f}")

# --------------------------
# **Questions:**
# --------------------------

# - Which pair is more similar? Why does this make sense?
    # Machine Learning and Reinforcement Learning are slightly more similar (0.6325 > 0.5919).
    # This makes sense because:
    # Reinforcement Learning is clearly defined as a machine learning approach.
    # The sentence contains strong direct relation to "machine learning".

# - What does the similarity score tell you about semantic relationships?
    # the similarity score tell how close two documents are in meaning
    # higher score -> more similar
    # lower score -> less similar

Similarity (ML vs DL): 0.5919
Similarity (ML vs RL): 0.6325


## Part 3: Building the Retrieval System

### Understanding the Importance

The retrieval system is the core of your pipeline. It takes a query and finds the most relevant documents from your knowledge base. The efficiency and accuracy of this component directly impact the performance of your entire RAG system.

**Why this matters:** A slow or inaccurate retrieval system will create bottlenecks and provide poor context to your language model, resulting in irrelevant or incorrect responses.


In [121]:
# -----------------------------------------------
### Task 3.1: Implement Basic Retrieval
# -----------------------------------------------

# Create a function that retrieves the top-k most similar documents for a given query.

def retrieve_documents(query, documents, embeddings, model, top_k=3):
    """
    Retrieve the most relevant documents for a query.
    
    Args:
        query: Search query string
        documents: List of document texts
        embeddings: Pre-computed document embeddings
        model: Embedding model
        top_k: Number of documents to retrieve
    
    Returns:
        List of tuples (document, similarity_score)
    """
    # Your code here
    # converting query into embedding
    query_embedding = model.encode(query)

    # empty list to store similarity score
    similarities=[]

    # Calculates similarity between query and each document
    for i, doc_embedding in enumerate(embeddings):

        # calculating cosine similarity
        score = cosine_similarity(query_embedding, doc_embedding)

        # store document and its similarity score
        similarities.append((documents[i], score))

   # sort documents by similarity score
    similarities.sort(key = lambda x: x[1], reverse = True)

    # return top_k results
    return similarities[:top_k]

# -----------------------------------
# **Testing retrieval system:**
# -----------------------------------

# user search query
query = "How do computers learn from data?"
# get top 3 most relevant documents
results = retrieve_documents(query, DOCUMENTS, embeddings, model, top_k=3)

# print query
print(f"Query: {query}\n")
# loop through result
for i, (doc, score) in enumerate(results, 1):
    # print similarity score
    print(f"Result {i} (Score: {score:.4f}):")
    # print document tex
    print(f"{doc}\n")

Query: How do computers learn from data?

Result 1 (Score: 0.5245):
Machine learning is a subset of artificial intelligence that enables systems to learn and improve from experience without being explicitly programmed.

Result 2 (Score: 0.5062):
Supervised learning involves training a model on labeled data, where the correct output is known for each input example.

Result 3 (Score: 0.4734):
Unsupervised learning involves training models on data without labeled responses, finding hidden patterns or structures in the data.



In [127]:
# -----------------------------------------------
### Task 3.2: Implement Retrieval with Threshold
# -----------------------------------------------

# Enhance your retrieval function to only return documents above a similarity threshold.

def retrieve_with_threshold(query, documents, embeddings, model, 
                           top_k=3, threshold=0.3):
    """
    Retrieve documents that meet a minimum similarity threshold.
    
    Args:
        query: Search query string
        documents: List of document texts
        embeddings: Pre-computed document embeddings
        model: Embedding model
        top_k: Maximum number of documents to retrieve
        threshold: Minimum similarity score
    
    Returns:
        List of tuples (document, similarity_score)
    """
    # Your code here
    # converting query into embedding
    query_embedding = model.encode(query)
    # empty list to store result
    results = [] 

    # comparing query with all documents
    for i, doc_embedding in enumerate(embeddings):
        # calculates similarity score
        score = cosine_similarity(query_embedding, doc_embedding)

        # keep the embeddings only if score meets the threshold
        if score >= threshold: #keeps only the relevant documents
            results.append((documents[i], score))

    # sorting by the highest similarity
    results.sort(key=lambda x: x[1], reverse=True)

    # return Top-k results
    return results[:top_k]
        

# -------------------------------------------
# **Experiment with different thresholds:**
# -------------------------------------------

queries = [
    "What is deep learning?",
    "How do neural networks work?",
    "Tell me about quantum computing"  # Out of domain query
]

for query in queries:
    print(f"\nQuery: {query}")
    results = retrieve_with_threshold(query, DOCUMENTS, embeddings, model, 
                                     top_k=3, threshold=0.3)
    print(f"Retrieved {len(results)} documents")
    if results:
        print(f"Top result (Score: {results[0][1]:.4f}): {results[0][0][:100]}...")

# --------------------------
# **Questions:**
# --------------------------
# - What happens when you query about topics not in your knowledge base?
    # If the asked topic is not in knowledge base then similarity score will be low, and threshold is high then system correctly shows 0 documents

# - How does the threshold help filter irrelevant results?
    # Threshold removes the documents with low similarity, improve answer quality and reduce hallucination risk.
    



Query: What is deep learning?
Retrieved 3 documents
Top result (Score: 0.7955): Deep learning is a type of machine learning based on artificial neural networks with multiple layers...

Query: How do neural networks work?
Retrieved 3 documents
Top result (Score: 0.7190): Neural networks are computing systems inspired by biological neural networks, consisting of intercon...

Query: Tell me about quantum computing
Retrieved 1 documents
Top result (Score: 0.3105): Neural networks are computing systems inspired by biological neural networks, consisting of intercon...



## Part 4: Optimizing the Retrieval Pipeline

### Understanding the Importance

As your knowledge base grows, retrieval performance becomes critical. You need efficient data structures and algorithms to handle thousands or millions of documents.

**Why this matters:** A retrieval system that works well with 10 documents might become unusable with 10,000. Learning optimization techniques early prevents scaling problems later.

In [128]:
# ---------------------------------------
### Task 4.1: Implement Batch Processing
# ---------------------------------------

# Create a function that efficiently handles multiple queries at once.

def batch_retrieve(queries, documents, embeddings, model, top_k=3):
    """
    Efficiently retrieve documents for multiple queries.
    
    Args:
        queries: List of query strings
        documents: List of document texts
        embeddings: Pre-computed document embeddings
        model: Embedding model
        top_k: Number of documents per query
    
    Returns:
        Dictionary mapping each query to its results
    """
    # Your code here
    # Hint: Create embeddings for all queries at once

    # dictionary to store results for each query
    results_dict = {}

# creates embeddings for all queries at once
    query_embeddings = model.encode(queries) # this is faster than encoding one by one

# loop through each query embedding
    for idx, query_embedding in enumerate(query_embeddings):
    # store similarity scores for this query
        similarities = []

    # compare query with all document embeddings
        for i, doc_embedding in enumerate(embeddings):
        #  calculate similarity score
            score = cosine_similarity(query_embedding, doc_embedding)
        # store document and its score
            similarities.append((documents[i], score))
    # sort document by highest similarity     
        similarities.sort(key= lambda x: x[1], reverse = True)

    # store top_k results for this query
        results_dict[queries[idx]]= similarities[:top_k]

 # return dictionary containing results for all queries    
    return results_dict   

# -----------------------------------------
# **Testing the class:**
# -----------------------------------------

queries = [
    "What is deep learning?",
    "How do computers learn?",
    "Explain neural networks"
]
results = batch_retrieve(queries, DOCUMENTS, embeddings, model, top_k=2)

# loop through each query result
for query, docs in results.items():
    print(f"\nQuery: {query}")

    for i, (doc, score) in enumerate(docs, 1):
        print(f"Result {i} (Score: {score:.4f})")
        print(f"{doc}\n")


Query: What is deep learning?
Result 1 (Score: 0.7955)
Deep learning is a type of machine learning based on artificial neural networks with multiple layers, allowing computers to learn complex patterns.

Result 2 (Score: 0.5616)
Machine learning is a subset of artificial intelligence that enables systems to learn and improve from experience without being explicitly programmed.


Query: How do computers learn?
Result 1 (Score: 0.4925)
Machine learning is a subset of artificial intelligence that enables systems to learn and improve from experience without being explicitly programmed.

Result 2 (Score: 0.4567)
Supervised learning involves training a model on labeled data, where the correct output is known for each input example.


Query: Explain neural networks
Result 1 (Score: 0.7010)
Neural networks are computing systems inspired by biological neural networks, consisting of interconnected nodes that process information.

Result 2 (Score: 0.6182)
Deep learning is a type of machine learn

In [129]:
# -----------------------------------------
### Task 4.2: Create a Retrieval Class
# -----------------------------------------


# Encapsulate your retrieval pipeline in a reusable class.

class RetrievalPipeline:
    """
    A complete retrieval pipeline for RAG systems.
    """
    
    def __init__(self, model_name='all-MiniLM-L6-v2'):
        """Initialize the pipeline with an embedding model."""
        # Your code here
        # load embedding model
        self.model = SentenceTransformer(model_name)

        # store documents
        self.documents = []

        # store document embeddings
        self.embeddings = None
    
    def add_documents(self, documents):
        """
        Add documents to the knowledge base.
        
        Args:
            documents: List of text documents
        """
        # Your code here
        # save documents
        self.documents = documents

        # create embeddings for all documents
        self.embeddings = self.model.encode(documents)

        # convert to numpy array for calculations
        self.embeddings = np.array(self.embeddings)
    
    def retrieve(self, query, top_k=3, threshold=0.0):
        """
        Retrieve relevant documents for a query.
        
        Args:
            query: Search query
            top_k: Number of results
            threshold: Minimum similarity score
        
        Returns:
            List of (document, score) tuples
        """
        # Your code here
        # convert query into embedding
        quer_embedding = self.model.encode(query)

        # store matching documents
        results = []

        # loop through all document embeddings
        for i, doc_embedding in enumerate(self.embeddings):
            # Calculate similarity score
            score = cosine_similarity(query_embedding, doc_embedding)

            # keep only if score meets threshold
            if score >= threshold:
                # store documents and score
                results.append((self.documents[i], score))
                
        # sort by highest similarity
        results.sort(key = lambda x: x[1], reverse = True)
        
        # return top_k results
        return results[:top_k]
    
    def get_stats(self):
        """Return statistics about the knowledge base."""
        # Your code here
        # count number of documents
        num_docs = len(self.documents)

        # get embedding dimentions
        embedding_dim = self.embeddings.shape[1] if self.embeddings is not None else 0

        # return statistics as dictionary
        return {
            "Number of Documents": num_docs,
            "Embedding Dimension": embedding_dim
        } 

# --------------------------
# **Testing the class:**
# --------------------------    


from documents import DOCUMENTS

# create pipeline object
pipeline = RetrievalPipeline()
# add documents and create embeddings
pipeline.add_documents(DOCUMENTS)

# printing stats about stored documents
print("Pipeline Statistics:")
print(pipeline.get_stats())

# example query
query = "What are neural networks?"
# retrieve top 2 relevant documents
results = pipeline.retrieve(query, top_k=2)

# print score and first 80 characters of document
print(f"\nQuery: {query}")
for doc, score in results:
    print(f"Score: {score:.4f} | {doc[:80]}...")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Pipeline Statistics:
{'Number of Documents': 10, 'Embedding Dimension': 384}

Query: What are neural networks?
Score: 0.7955 | Deep learning is a type of machine learning based on artificial neural networks ...
Score: 0.5616 | Machine learning is a subset of artificial intelligence that enables systems to ...


## Part 5: Evaluation and Analysis

### Understanding the Importance

You can't improve what you don't measure. Evaluation metrics help you understand how well your retrieval system performs and guide optimization efforts.

**Why this matters:** Without proper evaluation, you won't know if changes to your pipeline improve or degrade performance. In production systems, this can mean the difference between helpful and harmful AI responses.


In [130]:
# -----------------------------------------
### Task 5.1: Implement Relevance Metrics
# -----------------------------------------

# Create functions to measure retrieval quality.

# ------------------
# precsion
# ------------------
def calculate_precision_at_k(retrieved_docs, relevant_docs, k=3):
    """
    Calculate precision at k: proportion of retrieved docs that are relevant.
    
    Args:
        retrieved_docs: List of retrieved document indices
        relevant_docs: Set of relevant document indices
        k: Number of top results to consider
    
    Returns:
        Precision score
    """
    # Your code here
    # take only top k retrieved documents
    top_k_docs = retrieved_docs[:k]

    # count how many are relevant
    relevant_count = 0

    for doc in top_k_docs:
        if doc in relevant_docs:
            relevant_count += 1
    # precision formula
    precision = relevant_count / k

    return precision

# ------------------
# Recalll
# ------------------
def calculate_recall_at_k(retrieved_docs, relevant_docs, k=3):
    """
    Calculate recall at k: proportion of relevant docs that were retrieved.
    
    Args:
        retrieved_docs: List of retrieved document indices
        relevant_docs: Set of relevant document indices
        k: Number of top results to consider
    
    Returns:
        Recall score
    """
    # Your code here
    # take only top k retrieved documents
    top_k_docs = retrieved_docs[:k]

    # count how many are relevant
    relevant_count = 0

    for doc in top_k_docs:
        if doc in relevant_docs:
            relevant_count +=1

    # recall formula
    # avoid division by zero 
    if len(relevant_docs) == 0:
        return 0.0
    recall = relevant_count / len(relevant_docs)  
    return recall

# ------------------------
# **Testing the class:**
# ------------------------

# documents returned by system
retrieved_docs = [0,1,4]

# actual correct documents
relevant_docs = {0, 4}

# calculate precision 
precision = calculate_precision_at_k(retrieved_docs, relevant_docs, k=3)

# calculate recall
recall = calculate_recall_at_k(retrieved_docs, relevant_docs, k= 3)

# print precision score
print("Precsion: ", precision)

# print recall score
print("Recall: ", recall)


Precsion:  0.6666666666666666
Recall:  1.0


In [131]:
#--------------------------------------------
### Task 5.2: Create Test Cases
#--------------------------------------------

# Define test queries with known relevant documents.

# Test cases: (query, set of relevant document indices)
test_cases = [
    ("What is machine learning?", {0, 7, 8}),  # ML, supervised, unsupervised
    ("Explain neural networks", {1, 6, 9}),     # Deep learning, neural nets, transformers
    ("How does computer vision work?", {3}),    # Computer vision
]

def evaluate_retrieval(pipeline, test_cases, k=3):
    """
    Evaluate retrieval performance on test cases.
    
    Args:
        pipeline: RetrievalPipeline instance
        test_cases: List of (query, relevant_docs) tuples
        k: Number of results to retrieve
    
    Returns:
        Dictionary with average precision and recall
    """
    # Your code here
    # For each test case:
    # 1. Retrieve documents
    # 2. Get indices of retrieved documents
    # 3. Calculate precision and recall
    # 4. Average across all test cases
    
    # store sum of precision scores
    total_precision = 0

    # store sum of recall scores
    total_recall = 0

    # total number of test cases
    num_cases = len(test_cases)

    # loop through each test case
    for query, relevant_docs in test_cases:

        # retrieve top-k documents
        results = pipeline.retrieve(query, top_k =k)
        # store indices of retrieved documents
        retrieved_indices = []

        # loop through retrieved documents
        for doc, score in results:
            # finding index of document in knowledge base
            index = pipeline.documents.index(doc)
            # store index
            retrieved_indices.append(index)
        # calculate precision for this query
        precision = calculate_precision_at_k(retrieved_indices, relevant_docs, k)
        # Calculate recall for this query
        recall = calculate_recall_at_k(retrieved_indices, relevant_docs, k)
        
        # add to total precision
        total_precision += precision
        # add to total recall
        total_recall += recall
        
      # Calculate average precision
    avg_precision = total_precision / num_cases
     # Calculate average recall
    avg_recall = total_recall / num_cases

    # return evaluation results
    return {
        "Average Precision": avg_precision,
        "Average Recall": avg_recall
    }
        
    

In [132]:
#--------------------------------------------
### Task 5.3: Analyze Retrieval Performance
#--------------------------------------------

# Run evaluation and analyze the results.

# Your code to evaluate and print results

# analyze each query separately
for query, relevant_docs in test_cases:
    
    # retrieve top k documents
    results = pipeline.retrieve(query, top_k=3)
    
    # store retrieved indices
    retrieved_indices = []
    
    # extract indices of retrieved documents
    for doc, score in results:
        index = pipeline.documents.index(doc)
        retrieved_indices.append(index)
    
    # calculate precision
    precision = calculate_precision_at_k(retrieved_indices, relevant_docs, 3)
    
    # calculate recall
    recall = calculate_recall_at_k(retrieved_indices, relevant_docs, 3)
    # print query analysis
    print("\nquery:", query)
    print("precision:", precision)
    print("recall:", recall)
    
# ------------------------------------------------
# **Analysis Questions:**
# ------------------------------------------------
# - What is the average precision and recall of your system?
    #Ans: It depends on the output, but generally precision shows how many retrieved documents are correct and recall shows how many relavent documents are retrieved
        # if both close to 1 then system is performing well

# - Which queries perform best/worst? Why?
    #Ans: Best performing queries:
        # Clear and specific topics (like "computer vision")
        # Because only one document matches strongly
        #Worst performing queries:
        # Broad topics (like "machine learning")
        # Because multiple related documents exist
        # Harder to rank perfectly
    
# - How could you improve retrieval quality?
    #Ans: By using better embedding models, increasing document chunk quality, adjusting similarity threshold, increasing top_k and using ANN for better nearest neighbor search
    


query: What is machine learning?
precision: 0.3333333333333333
recall: 0.3333333333333333

query: Explain neural networks
precision: 0.3333333333333333
recall: 0.3333333333333333

query: How does computer vision work?
precision: 0.3333333333333333
recall: 1.0


## Part 6: Real-World Application

### Understanding the Importance

This final section bridges the gap between learning and application. You'll work with a more realistic scenario that includes challenges you'll face in production systems.

**Why this matters:** Textbook examples rarely capture the messiness of real data. Learning to handle longer documents, mixed content types, and edge cases prepares you for actual RAG implementations.

In [134]:
# ------------------------------------------------
### Task 6.1: Extended Knowledge Base
# ------------------------------------------------
# Create a more complex dataset with longer documents.


EXTENDED_DOCS = [
    """
    Artificial Intelligence (AI) is transforming industries worldwide. Machine learning, 
    a subset of AI, enables systems to learn from data without explicit programming. 
    Companies use ML for predictive analytics, recommendation systems, and automation. 
    The field has grown exponentially with the availability of big data and computational power.
    """,
    """
    Deep learning has revolutionized computer vision and natural language processing. 
    Convolutional neural networks (CNNs) excel at image recognition tasks, achieving 
    human-level performance in many cases. Applications include medical imaging, 
    autonomous vehicles, and facial recognition systems.
    """,
    """
    Natural language processing enables machines to understand human language. Modern NLP 
    uses transformer architectures like BERT and GPT. These models power chatbots, 
    translation services, and search engines. NLP is crucial for human-computer interaction.
    """
    
]

In [135]:
# create new pipeline instance
extended_pipeline = RetrievalPipeline()

# add extended documents
extended_pipeline.add_documents(EXTENDED_DOCS)

# print statistics
print("extended pipeline stats:")
print(extended_pipeline.get_stats())

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


extended pipeline stats:
{'Number of Documents': 3, 'Embedding Dimension': 384}


In [136]:
# ------------------------------------------
### Task 6.2: Handle Multi-Document Queries
# ------------------------------------------

# Implement a function that retrieves and combines information from multiple documents.

def multi_document_retrieve(query, pipeline, top_k=5, combine=True):
    """
    Retrieve and optionally combine information from multiple documents.
    
    Args:
        query: Search query
        pipeline: RetrievalPipeline instance
        top_k: Number of documents to retrieve
        combine: Whether to combine retrieved documents
    
    Returns:
        Combined context string or list of documents
    """
    # Your code here
    # retrieve top_k relevant documents
    results = pipeline.retrieve(query, top_k = top_k)

    # extract only document texts (ignore scores)
    documents = [doc for doc, score in results]
    # if combine is true, merge all documents into one context
    if combine:
        # join documents with separator for clarity
        combined_context = "\n\n".join(documents)

        # return single combined string
        return combined_context
    else:
        # return list of separate documents
        return documents



In [140]:
# ------------------------------------------------
### Task 6.3: Build a Complete RAG Retrieval Demo
# ------------------------------------------------

# Create a simple interactive demo of your retrieval system.


def interactive_retrieval_demo(pipeline):
    """
    Run an interactive demo of the retrieval system.
    """
    print("RAG Retrieval System Demo")
    print("=" * 50)
    print("Enter your queries (type 'quit' to exit)")
    print("=" * 50)
    
    # Your code here
    # Allow users to:
    # 1. Enter queries
    # 2. See retrieved documents with scores
    # 3. View statistics

    # starting interaction loop
    while True:
        # take user input
        query = input("\nEnter query: ").strip()
        # skip empty input
        if not query:
            print("Please enter a valid query.")
            continue
        
        # exit condition
        if query.lower() == "quit":
            print("Exiting demo...")
            break
        
        # show statistics
        if query.lower() == "stats":
            print("\nPipeline Statistics:")
            print(pipeline.get_stats())
            continue
        
        # retrieve documents
        results = pipeline.retrieve(query, top_k=3)        
        # check if nothing retrieved
        if not results:
            print("\nNo relevant documents found.")
            continue
        
        print("\nTop Retrieved Documents:")        
        for i, (doc, score) in enumerate(results, 1):
            print(f"\nResult {i}")
            print(f"Score: {score:.4f}")
            
            # handle both string docs and metadata docs
            if isinstance(doc, dict):
                print(f"Content: {doc['text'][:200]}...")
            else:
                print(f"Content: {doc[:200]}...")

# create pipeline
pipeline = RetrievalPipeline()
# add documents
pipeline.add_documents(EXTENDED_DOCS)
# check stats
print(pipeline.get_stats())
# run demo
interactive_retrieval_demo(pipeline)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


{'Number of Documents': 3, 'Embedding Dimension': 384}
RAG Retrieval System Demo
Enter your queries (type 'quit' to exit)



Enter query:  What is deep learning?



Top Retrieved Documents:

Result 1
Score: 0.6447
Content: 
    Deep learning has revolutionized computer vision and natural language processing. 
    Convolutional neural networks (CNNs) excel at image recognition tasks, achieving 
    human-level performanc...

Result 2
Score: 0.5044
Content: 
    Artificial Intelligence (AI) is transforming industries worldwide. Machine learning, 
    a subset of AI, enables systems to learn from data without explicit programming. 
    Companies use ML fo...

Result 3
Score: 0.3265
Content: 
    Natural language processing enables machines to understand human language. Modern NLP 
    uses transformer architectures like BERT and GPT. These models power chatbots, 
    translation services...



Enter query:  quit


Exiting demo...


## Challenge Tasks

### Challenge 1: Implement Re-ranking

After initial retrieval, re-rank results using a different strategy (e.g., keyword matching combined with semantic similarity).

In [141]:
# Creating Keyword Matching Function
def keyword_overlap_score(query, document):
    
    # convert to lowercase
    query_words = set(query.lower().split())
    doc_words = set(document.lower().split())
    
    # count common words
    common_words = query_words.intersection(doc_words)
    
    # calculate overlap ratio
    score = len(common_words) / len(query_words)
    
    return score

# Hybrid Re-ranking Function
def hybrid_retrieve(query, pipeline, top_k=5, alpha=0.7):
    """
    alpha controls weight of semantic similarity.
    final_score = alpha * semantic_score + (1-alpha) * keyword_score
    """
    
    # get initial semantic results
    initial_results = pipeline.retrieve(query, top_k=top_k)
    
    reranked_results = []
    
    for doc, semantic_score in initial_results:
        
        # calculate keyword score
        keyword_score = keyword_overlap_score(query, doc)
        
        # combine both scores
        final_score = alpha * semantic_score + (1 - alpha) * keyword_score
        
        reranked_results.append((doc, final_score))
    
    # sort using combined score
    reranked_results.sort(key=lambda x: x[1], reverse=True)
    
    return reranked_results
 
#-----------------------------------------
# Example Test
#-----------------------------------------

query = "How does computer vision work?"

results = hybrid_retrieve(query, pipeline, top_k=3)

for i, (doc, score) in enumerate(results, 1):
    print(f"\nResult {i}")
    print(f"Combined Score: {score:.4f}")
    print(doc[:150], "...")                                                                     


Result 1
Combined Score: 0.5713

    Deep learning has revolutionized computer vision and natural language processing. 
    Convolutional neural networks (CNNs) excel at image recogn ...

Result 2
Combined Score: 0.3531

    Artificial Intelligence (AI) is transforming industries worldwide. Machine learning, 
    a subset of AI, enables systems to learn from data with ...

Result 3
Combined Score: 0.2286

    Natural language processing enables machines to understand human language. Modern NLP 
    uses transformer architectures like BERT and GPT. Thes ...


### Challenge 2:
Add Metadata Filtering Extend your pipeline to support filtering documents by metadata (e.g., date, category, author).

In [142]:
class RetrievalPipeline:
    
    def __init__(self, model_name='all-MiniLM-L6-v2'):
        
        # load embedding model
        self.model = SentenceTransformer(model_name)
        
        # store documents (with metadata)
        self.documents = []
        
        # store embeddings
        self.embeddings = None
    
    
    def add_documents(self, documents):
        
        # store full documents including metadata
        self.documents = documents
        
        # extract text field for embedding
        texts = [doc["text"] for doc in documents]
        
        # create embeddings for all documents
        self.embeddings = self.model.encode(texts)
    
    
    def retrieve(self, query, top_k=3, filters=None):
        
        # convert query to embedding
        query_embedding = self.model.encode(query)
        
        # store retrieval results
        results = []
        
        # loop through documents
        for i, doc in enumerate(self.documents):
            
            # apply metadata filtering if filters provided
            if filters:
                # assume document matches
                match = True
                # check each filter condition
                for key, value in filters.items():
                    # if metadata does not match, skip document
                    if doc.get(key) != value:
                        match = False
                        break
                
                # skip non-matching documents
                if not match:
                    continue
            
            # calculate similarity score
            score = cosine_similarity(query_embedding, self.embeddings[i])
            # store text and score
            results.append((doc["text"], score))
        # sort results by similarity (highest first)
        results.sort(key=lambda x: x[1], reverse=True)
        
        # return top_k results
        return results[:top_k]

DOCUMENTS_WITH_META = [
    {"text": "Machine learning enables systems to learn from data.", "category": "ML", "author": "Alice", "year": 2020},
    {"text": "Deep learning uses neural networks with many layers.", "category": "DL", "author": "Bob", "year": 2021},
    {"text": "Computer vision helps machines understand images.", "category": "CV", "author": "Alice", "year": 2019}
]
    
# create pipeline
pipeline = RetrievalPipeline()

# add documents with metadata
pipeline.add_documents(DOCUMENTS_WITH_META)

# retrieve only ML category
results = pipeline.retrieve(
    query="learning systems",
    top_k=2,
    filters={"category": "ML"}
)

# print results
for doc, score in results:
    print(score, "|", doc)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


0.5446004 | Machine learning enables systems to learn from data.


### Challenge 3: Implement Hybrid Search

Combine dense retrieval (embeddings) with sparse retrieval (keyword-based like BM25) for improved results.

In [152]:
def keyword_score(query, document):
    
    # convert query to lowercase words
    query_words = query.lower().split()
    
    # convert document to lowercase words
    doc_words = document.lower().split()
    
    # count matching words
    match_count = 0
    
    for word in query_words:
        if word in doc_words:
            match_count += 1
    
    # normalize score
    return match_count / len(query_words)

# Hybrid Search Function
def hybrid_search(query, pipeline, top_k=3, alpha=0.7):
    """
    alpha controls weight of semantic similarity
    final_score = alpha * dense_score + (1 - alpha) * sparse_score
    """
    
    # convert query to embedding
    query_embedding = pipeline.model.encode(query)
    
    # store hybrid results
    results = []
    
    # loop through all documents
    for i, doc in enumerate(pipeline.documents):
        
        # get document text
        text = doc["text"] if isinstance(doc, dict) else doc
        
        # calculate dense similarity
        dense_score = cosine_similarity(query_embedding, pipeline.embeddings[i])
        
        # calculate sparse keyword score
        sparse_score = keyword_score(query, text)
        
        # combine both scores
        final_score = alpha * dense_score + (1 - alpha) * sparse_score
        
        # store text and combined score
        results.append((text, final_score))
    
    # sort by final score
    results.sort(key=lambda x: x[1], reverse=True)
    
    # return top_k results
    return results[:top_k]

# Example Usage
# run hybrid search
results = hybrid_search("computer vision systems", pipeline, top_k=3)

# print results
for doc, score in results:
    print(score, "|", doc[:100], "...")

0.7038358807563783 | Computer vision helps machines understand images. ...
0.30703217089176177 | Machine learning enables systems to learn from data. ...
0.19059577584266663 | Deep learning uses neural networks with many layers. ...


In [ ]:
## Questions

1. What are the key trade-offs between chunk size and retrieval quality?

Ans: Small chunks -> more precise results, but may lose context.
     Large chunks -> better context, but may include irrelevant information.
   
2. How would you handle documents in multiple languages

Ans: By using multilingual embedding models, detect language before processing.
    By storing language metadata and filter if needed, Normalize text properly (encoding, tokenization) 

3. What strategies could you use to update your knowledge base without recomputing all embeddings?

Ans:By Embedding only new or changed documents,not recomputing old embeddings and using incremental indexing.

4. How would you monitor retrieval quality in a production system?
Ans: By tracking precision and recall, monitor failed queries and collect user feedback.

5. What are the computational costs of your pipeline, and how would you optimize for large-scale deployment?
Ans: Costs:
        Embedding generation (model inference)
        Similarity computation (vector comparisons)

        Optimizations:
        Use ANN (Approximate Nearest Neighbor)
        Batch processing for queries
        Store embeddings in vector database
   
   
   